# Avaliação do modelo especialista: Alergia e Frieira

## Bibliotecas utilizadas

In [2]:
import numpy as np
import os
import re
from matplotlib import pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

## Definição das classes

In [3]:
actions = ['Alergia', 'Frieira']
num_frames = 30

#Número que representa cada classe
label_map = {label: num for num, label in enumerate(actions)}

#Nomes das classes nas figuras
labels_plot = ['Alergia', 'Frieira']

## Parâmetros e pastas

In [4]:
weights_dir = "../models/models/ale_fri"
results_dir = "../results/ale_fri"
os.makedirs(results_dir, exist_ok=True)

usar_latex = True #Colocar False se não tiver o Latex instalado

## Carregamento dos dados de validação

In [5]:
#Cada sujeito é um fold da validação cruzada (leave-one-subject-out)
subject_names = ['AL','e0','e1','e2','e3','ME','MV','RO','SA','MI']

#Dados de validação de cada sujeito
#X: janelas de keypoints dos vídeos
#y: rótulos dos vídeos
subject_data = {s: {'X': [], 'y': []} for s in subject_names}

pad = len(str(num_frames - 1)) #Largura do zero à esquerda no nome dos arquivos (00.npy, 01.npy, ...)

#Dados de validação: gerados no models/ale_fri_model.ipynb (detecção do início do movimento nos vídeos originais)
for action in actions:
    action_dir = os.path.join('../dataset_ale_fri/val', action)
    for video_folder in os.listdir(action_dir):
        #Identifica o sujeito pelo nome da pasta (ex.: Alergia_AL_ALE_AL_01 -> AL)
        m = re.search(r'_(%s)_' % '|'.join(subject_names), video_folder)
        if not m:
            raise ValueError(f"Não achou sujeito em {video_folder}")
        subj = m.group(1)

        video_dir = os.path.join(action_dir, video_folder)
        window = []
        for frame_num in range(num_frames):
            fname = f"{frame_num:0{pad}d}.npy"
            fpath = os.path.join(video_dir, fname)
            if not os.path.exists(fpath):
                break
            res = np.load(fpath)
            window.append(res)

        if len(window) == num_frames:
            label = label_map[action]
            subject_data[subj]['X'].append(window)
            subject_data[subj]['y'].append(label)

#Converte para array e transforma os rótulos em one-hot
for s in subject_names:
    subject_data[s]['X'] = np.array(subject_data[s]['X'])
    subject_data[s]['y'] = to_categorical(subject_data[s]['y'], num_classes=len(actions))

## Funções auxiliares

In [6]:
def class_acc(true, pred, labels):
    #Retorna a acurácia de cada classe
    acc = {}
    for i, lbl in enumerate(labels):
        idx = np.where(true == i)[0]
        acc[lbl] = np.mean(pred[idx] == true[idx]) if len(idx) > 0 else 0
    return acc


labels = list(label_map.keys())


def salva_matriz(cm, caminho):
    #Salva em PDF a matriz de confusão normalizada por linha, com a porcentagem e a quantidade de amostras em cada célula
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    #Porcentagens com duas casas decimais e vírgula (ex.: 76,00)
    percent_str = np.vectorize(lambda x: f"{x:0.2f}".replace(".", ","))(np.round(cm_norm * 100, 2))

    #Símbolo de porcentagem (com LaTeX é preciso usar a barra)
    simbolo = "\\%" if usar_latex else "%"

    #Texto de cada célula: porcentagem e quantidade de amostras (ex.: 76,00% e 38)
    annot_text = np.empty_like(percent_str, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot_text[i, j] = f"{percent_str[i, j]}{simbolo}\n{cm[i, j]}"

    #Fontes LaTeX nas figuras (requer LaTeX instalado)
    with plt.rc_context({"text.usetex": usar_latex, "font.family": "serif"}):
        plt.figure(figsize=(5,5))

        ax = sns.heatmap(
            cm_norm,
            annot=annot_text,
            fmt='',
            cmap='Blues',
            xticklabels=labels_plot,
            yticklabels=labels_plot,
            cbar=False,
            annot_kws={"size": 20},
            square=True
        )

        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=20)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

        plt.tight_layout()
        plt.savefig(caminho)
        plt.close()

## Avaliação dos 10 folds

In [7]:
yhat_all = []
ytrue_all = []
fold_accs = [] #Acurácia geral de cada fold
class_fold_results = [] #Acurácia por classe de cada fold

for i, val_subj in enumerate(subject_names):
    X_val = subject_data[val_subj]['X']
    y_val = subject_data[val_subj]['y']

    #O modelo do fold i foi treinado sem o sujeito i
    model = load_model(f'{weights_dir}/model_fold_{i+1}.keras')

    y_pred = np.argmax(model.predict(X_val), axis=1)
    y_true = np.argmax(y_val, axis=1)

    yhat_all.append(y_pred)
    ytrue_all.append(y_true)

    acc_fold = np.mean(y_pred == y_true)
    fold_accs.append(acc_fold)
    print(f"Fold {i+1} – {val_subj}: acc = {acc_fold:.4f}")

    cls_acc = class_acc(y_true, y_pred, labels)
    class_fold_results.append(cls_acc)

    for lbl, a in cls_acc.items():
        print(f"  {lbl}: {a:.4f}")

#Junta as predições dos 10 folds
y_true_final = np.concatenate(ytrue_all)
y_pred_final = np.concatenate(yhat_all)

#Acurácia geral, juntando as predições de todos os folds
acc_global = np.mean(y_pred_final == y_true_final)
print(f"\nAcurácia geral (predições dos 10 folds juntas): {acc_global:.4f}")

#Média e desvio padrão entre os folds, em porcentagem
print(f"\n{'':10s} | média folds | desvio folds")
for lbl in labels:
    valores = [d[lbl] * 100 for d in class_fold_results]
    print(f"{lbl:10s} | {np.mean(valores):11.2f} | {np.std(valores):12.2f}")
print(f"{'Geral':10s} | {np.mean(fold_accs) * 100:11.2f} | {np.std(fold_accs) * 100:12.2f}")

E0000 00:00:1791808449.656464   10115 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791808449.668562   10115 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step
Fold 1 – AL: acc = 0.9000
  Alergia: 1.0000
  Frieira: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
Fold 2 – e0: acc = 0.9000
  Alergia: 1.0000
  Frieira: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step
Fold 3 – e1: acc = 0.9000
  Alergia: 1.0000
  Frieira: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
Fold 4 – e2: acc = 1.0000
  Alergia: 1.0000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
Fold 5 – e3: acc = 1.0000
  Alergia: 1.0000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step
Fold 6 – ME: acc = 1.0000
  Alergia: 1.0000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
Fold 7 – MV: acc = 1.0000
  Alergia: 1.0000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
Fold 8 – RO: acc = 0.7000
  Alergia: 0.4000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
Fold 9 – SA: acc = 1.0000
  Alergia: 1.0000
  Frieira: 1.0000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step
Fold 10 – MI: acc = 1.0000
  Alergia: 1.0000
  Friei

## Matriz de confusão

In [8]:
cm = confusion_matrix(y_true_final, y_pred_final)
salva_matriz(cm, os.path.join(results_dir, "matriz_ale_fri.pdf"))

## Métricas por classe

In [9]:
print("\n=========== MÉTRICAS FINAIS POR CLASSE ===========\n")

precisions = precision_score(y_true_final, y_pred_final, average=None, zero_division=0)
recalls = recall_score(y_true_final, y_pred_final, average=None, zero_division=0)
f1s = f1_score(y_true_final, y_pred_final, average=None, zero_division=0)
f2s = fbeta_score(y_true_final, y_pred_final, beta=2, average=None, zero_division=0)

for i, lbl in enumerate(labels):
    print(f"Classe: {lbl}")
    print(f"  Precisão:  {precisions[i]:.4f}")
    print(f"  Recall:    {recalls[i]:.4f}")
    print(f"  F1-score:  {f1s[i]:.4f}")
    print(f"  F2-score:  {f2s[i]:.4f}")
    print()


=========== MÉTRICAS FINAIS POR CLASSE ===========

Classe: Alergia
  Precisão:  0.9400
  Recall:    0.9400
  F1-score:  0.9400
  F2-score:  0.9400

Classe: Frieira
  Precisão:  0.9400
  Recall:    0.9400
  F1-score:  0.9400
  F2-score:  0.9400



## Métricas macro

In [10]:
precision_macro = precision_score(y_true_final, y_pred_final, average='macro', zero_division=0)
recall_macro = recall_score(y_true_final, y_pred_final, average='macro', zero_division=0)
f1_macro = f1_score(y_true_final, y_pred_final, average='macro', zero_division=0)
f2_macro = fbeta_score(y_true_final, y_pred_final, beta=2, average='macro', zero_division=0)

print(f"  Precisão:  {precision_macro:.4f}")
print(f"  Recall:    {recall_macro:.4f}")
print(f"  F1-score:  {f1_macro:.4f}")
print(f"  F2-score:  {f2_macro:.4f}")
print()

  Precisão:  0.9400
  Recall:    0.9400
  F1-score:  0.9400
  F2-score:  0.9400

